In [2]:
import dash
from dash import dcc, html, Input, Output, State, dash_table
import pandas as pd
import joblib
import plotly.express as px

# Initialize Dash app
app = dash.Dash(__name__)
server = app.server  # for deployment

# Load trained model
model = joblib.load("models/logistic_model.pkl")

# App layout
app.layout = html.Div([
    html.H1("Fraud Detection Dashboard", style={"textAlign": "center"}),

    dcc.Upload(
        id="upload-data",
        children=html.Button("Upload CSV File"),
        multiple=False
    ),

    html.Div(id="output-table"),
    html.Div(id="output-graph"),
    html.Div(id="output-predictions")
])

In [3]:
import base64
import io

@app.callback(
    [Output("output-table", "children"),
     Output("output-graph", "children"),
     Output("output-predictions", "children")],
    [Input("upload-data", "contents")],
    [State("upload-data", "filename")]
)
def process_file(contents, filename):
    if contents is None:
        return None, None, None

    # Decode uploaded file
    content_type, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)
    df = pd.read_csv(io.StringIO(decoded.decode('utf-8')))

    # Predict using trained model
    X = df.drop("Class", axis=1, errors="ignore")
    predictions = model.predict(X)
    df["Prediction"] = predictions

    # Table
    table = dash_table.DataTable(
        data=df.head(10).to_dict("records"),
        columns=[{"name": i, "id": i} for i in df.columns],
        style_table={"overflowX": "auto"},
        page_size=10
    )

    # Graph
    fig = px.histogram(df, x="Amount", color="Prediction", nbins=50)
    graph = dcc.Graph(figure=fig)

    # Summary
    fraud_count = df["Prediction"].sum()
    total = len(df)
    summary = html.Div([
        html.H4(f"Detected Fraudulent Transactions: {fraud_count} / {total}")
    ])

    return table, graph, summary

In [5]:
if __name__ == "__main__":
    app.run(debug=True)